# Score-Based Data Assimilation Activity

## Downloads and settings




In [ ]:
pip install zuko POT

In [ ]:
!git clone https://github.com/pdenailly/score-da-colab.git
%cd score-da-colab
import sys
sys.path.append(".")

In [ ]:
#%load_ext autoreload
#%autoreload 2

import itertools
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import argparse

from pathlib import Path
from typing import *
import os

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from sda.mcs import *
from sda.score import *
from sda.utils import *

from utils import *

## Lorentz 1963 system

### Generation process

### Data Generation Process

The Lorenz 1963 system is simulated to generate chaotic trajectories:

1. **Markov Chain**: Creation of a `NoisyLorenz63` model with time step `dt=0.025`
2. **Initial Conditions**: Sampling of 1024 starting points from the prior distribution
3. **Simulation**: Generation of long trajectories (1024 time steps)
4. **Preprocessing**: Data normalization for score model training
5. **Splitting**: Separation into training (80%), validation (10%), and test (10%) sets

The final data have the shape `(n_trajectories, temporal_length, 3_variables)`.

In [ ]:
chain = make_chain()

x = chain.prior((1024,))
x = chain.trajectory(x, length=1024, last=True)
x = chain.trajectory(x, length=1024)
x = chain.preprocess(x)
x = x.transpose(0, 1)

i = int(0.8 * len(x))
j = int(0.9 * len(x))

train_data = x[:i]
valid_data = x[i:j]
test_data = x[j:]

### Visualization of 15 train trajectories

### Visualization of Training Trajectories

This visualization shows 15 trajectories of the Lorenz system as 3D point clouds:

- **Axes**: Normalized variables a, b, c (corresponding to the physical system coordinates x, y, z)
- **Representation**: Each trajectory is a point cloud with a different color
- **Scale**: Data are preprocessed in the model’s training space
- **Objective**: To verify the diversity of trajectories and the chaotic structure of the attractor

The famous Lorenz “butterfly” should be visible in the point distribution.

In [ ]:

# Number of trajectories to plot
n_trajectories = 15

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Colors for different trajectories
colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'olive', 'cyan']

for i in range(n_trajectories):
    traj = train_data[:, i, :]
    ax.scatter(traj[:, 0], traj[:, 1], traj[:, 2],
               c=colors[i % len(colors)],
               s=1,
               alpha=0.6,
               label=f'Trajectory {i+1}')

ax.set_title('Lorentz system trajectories', fontsize=14)
ax.set_xlabel('Variable a')
ax.set_ylabel('Variable b')
ax.set_zlabel('Variable c')

ax.legend()
ax.grid(True, alpha=0.3)

# Limits for visulization
ax.set_xlim([train_data[:, :, 0].min(), train_data[:, :, 0].max()])
ax.set_ylim([train_data[:, :, 1].min(), train_data[:, :, 1].max()])
ax.set_zlim([train_data[:, :, 2].min(), train_data[:, :, 2].max()])

plt.tight_layout()
plt.show()

print(f"Number of plotted trajectories: {n_trajectories}")
print(f"Points per trajectory: {train_data.shape[0]}")
print(f"Train data shape: {train_data.shape}")

## Score model training

### In-Memory Dataset for Training

The `TrajectoryDataset` class from the original SDA package expects an HDF5 file on disk. It provides training and validation data to the score model. To work directly with the data generated in memory within this notebook, we create an `InMemoryTrajectoryDataset` wrapper that:

- Accepts PyTorch tensors directly as input
- Applies temporal windowing and flattening as in the original implementation
- Enables training without saving/loading files

In [ ]:
from torch.utils.data import Dataset

class InMemoryTrajectoryDataset(Dataset):
    def __init__(self, data, window=None, flatten=False):
        self.data = data
        self.window = window
        self.flatten = flatten

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]

        if self.window is not None:
            start = torch.randint(0, len(x) - self.window + 1, size=())
            x = torch.narrow(x, dim=0, start=start.item(), length=self.window)

        if self.flatten:
            x = x.flatten(0, 1)

        return x, {}


### Local score model with window size 5 (k=2)

A first training using a score model (time-conditioned MLP with Fourier embeddings) is performed with a window size of 5, which corresponds to accounting for k=2. The training is therefore carried out within a very local neighborhood.

In [ ]:
LOCAL_CONFIG = {
    # Architecture
    'window': 5,
    'embedding': 32,
    'width': 256,
    'depth': 5,
    'activation': 'SiLU',
    # Training
    'epochs': 2048,
    'batch_size': 64,
    'optimizer': 'AdamW',
    'learning_rate': 1e-3,
    'weight_decay': 1e-3,
    'scheduler': 'linear',
}


# Network
window = LOCAL_CONFIG['window']
score_2 = make_local_score(**LOCAL_CONFIG)
sde_2 = VPSDE(score_2.kernel, shape=(window * 3,)).cuda()

# Data
trainset = InMemoryTrajectoryDataset(train_data, window=window, flatten=True)
validset = InMemoryTrajectoryDataset(valid_data, window=window, flatten=True)

print(f"Training samples: {len(trainset)}")
print(f"Validation samples: {len(validset)}")

# Training
generator = loop(
    sde_2,
    trainset,
    validset,
    **LOCAL_CONFIG,
    device='cuda',
)

for loss_train, loss_valid, lr in generator:
    pass

print("Training finished!")


### Score model with window size 9 (k=4)

A second training using a score model is performed with a window size of 9, which corresponds to accounting for k=4.

In [ ]:
LOCAL_CONFIG = {
    # Architecture
    'window': 9,
    'embedding': 32,
    'width': 256,
    'depth': 5,
    'activation': 'SiLU',
    # Training
    'epochs': 2048,
    'batch_size': 64,
    'optimizer': 'AdamW',
    'learning_rate': 1e-3,
    'weight_decay': 1e-3,
    'scheduler': 'linear',
}


# Network
window = LOCAL_CONFIG['window']
score_4 = make_local_score(**LOCAL_CONFIG)
sde_4 = VPSDE(score_4.kernel, shape=(window * 3,)).cuda()

# Data
trainset = InMemoryTrajectoryDataset(train_data, window=window, flatten=True)
validset = InMemoryTrajectoryDataset(valid_data, window=window, flatten=True)

print(f"Training samples: {len(trainset)}")
print(f"Validation samples: {len(validset)}")

# Training
generator = loop(
    sde_4,
    trainset,
    validset,
    **LOCAL_CONFIG,
    device='cuda',
)

for loss_train, loss_valid, lr in generator:
    pass

print("Training finished!")


## Evaluation

Creation of two types of observations: a low-frequency, weakly noisy one (y_lo) and a high-frequency, strongly noisy one (y_hi).

In [ ]:
test = test_data[:, :65]
y_lo = torch.normal(test[:, ::8, :1], 0.05)
y_hi = torch.normal(test[:, :, :1], 0.25)

### Samples generation from observations

In the code below, we generate samples from a set of observations (y_hi or y_lo) and a model (score_2 or score_4).

## 🔄 Sampling from Partial Observations

### `sample_from_partial_observation`

This function generates samples from a score-based generative model conditioned on partial observations.

It performs conditional sampling using a stochastic differential equation (SDE) framework, where the model is guided by observed data (`y`) that can be either low-frequency (`y_lo`) or high-frequency (`y_hi`).

### Key steps:

- **Device handling**: Moves the score model and observations to the correct device.
- **Frequency-dependent observation model**:
  - `freq='lo'`: sparse observations with lower noise (`sigma=0.05`, subsampling `step=8`)
  - `freq='hi'`: dense observations with higher noise (`sigma=0.25`, full resolution `step=1`)
- **Observation operator**:
  - A linear operator `A(x)` extracts observed components from the full state.
- **Conditional SDE construction**:
  - A `VPSDE` is built using a Gaussian observation model and the learned score function.
- **Sampling process**:
  - The model generates `n_samples` trajectories using a sampler with:
    - `steps`: number of integration steps
    - `corrections`: optional Langevin correction steps
    - `tau`: noise scaling parameter
- **Post-processing**:
  - Samples are transformed back to physical space using a `chain.postprocess` function.

### Outputs:
- `x_preprocessed`: raw generated samples
- `x_postprocessed`: physically reconstructed trajectories
- `y`: observed data used for conditioning

In [ ]:
def sample_from_partial_observation(
    score,
    y,
    freq='hi',
    corrections=0,
    n_samples=256,
    steps=64,
    tau=0.25,
    device=None,
):
    chain = make_chain()

    if device is None:
        device = next(score.parameters()).device

    y = y.to(device)
    if freq == 'lo':
        sigma, step = 0.05, 8
    else:
        sigma, step = 0.25, 1

    sde = VPSDE(
        GaussianScore(
            y=y,
            A=lambda x: x[..., ::step, :1],
            std=sigma,
            sde=VPSDE(score, shape=()),
            gamma=3e-2,
        ),
        shape=(65, 3),
    ).to(device)

    x_preprocessed = sde.sample((n_samples,), steps=steps, corrections=corrections, tau=tau).cpu()
    x_postprocessed = chain.postprocess(x_preprocessed)
    return x_preprocessed, x_postprocessed, y.cpu()


def plot_partial_reconstructions(
    x_samples,
    y_obs,
    x_true=None,
    n_plot=5,
    freq='hi',
):
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D

    mean_sample = x_samples.mean(dim=0)
    x_plot = x_samples[:n_plot]

    fig = plt.figure(figsize=(18, 6))

    ax1 = fig.add_subplot(1, 3, 1, projection='3d')
    ax1.plot(mean_sample[:, 0], mean_sample[:, 1], mean_sample[:, 2], color='blue', label='Mean sample')
    if x_true is not None:
        ax1.plot(x_true[:, 0], x_true[:, 1], x_true[:, 2], color='black', linestyle='--', label='Truth')
    ax1.set_title('Built mean trajectory')
    ax1.set_xlabel('a')
    ax1.set_ylabel('b')
    ax1.set_zlabel('c')
    ax1.legend()

    ax2 = fig.add_subplot(1, 3, 2)
    for k in range(min(n_plot, len(x_plot))):
        ax2.plot(x_plot[k, :, 0].numpy(), alpha=0.5)
    if x_true is not None:
        ax2.plot(x_true[:, 0].numpy(), color='black', linestyle='--', linewidth=2, label='Truth')
    ax2.scatter(torch.arange(len(y_obs)) * (8 if freq == 'lo' else 1), y_obs[:, 0].numpy(), color='red', s=20, label='Observations')
    ax2.set_title('Projection on a and partial observations')
    ax2.set_xlabel('Temporal index')
    ax2.set_ylabel('a')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    ax3 = fig.add_subplot(1, 3, 3)
    ax3.plot(mean_sample[:, 0].numpy(), label='Mean a')
    if x_true is not None:
        ax3.plot(x_true[:, 0].numpy(), linestyle='--', color='black', label='True a')
    ax3.scatter(torch.arange(len(y_obs)) * (8 if freq == 'lo' else 1), y_obs[:, 0].numpy(), color='red', s=20, label='Obs a')
    ax3.set_title('Comparison on coordiante a')
    ax3.set_xlabel('Temporal index')
    ax3.set_ylabel('a')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


In [ ]:
indice_traj = 1
freq = 'hi'
y = y_lo if freq == 'lo' else y_hi
score_ = score_2

# Example : Conditional generation and visu
x_preprocessed, x_postprocessed, y_obs = sample_from_partial_observation(
    score=score_,
    y=y[indice_traj],
    freq=freq,
    corrections=5,
    n_samples=256,
    steps=64,
    tau=0.25,
)

x_true = test[1]

print(f"Generated samples : {x_preprocessed.shape}")
plot_partial_reconstructions(
    x_preprocessed,
    y_obs,
    x_true=x_true,
    n_plot=5,
    freq=freq,
)


### Evaluation function

Finaly, we evaluate the quality of a model under partial and noisy observations.  
The following function generates samples conditioned on observations `y`, then measures both:

- the consistency of the generated trajectories with the prior dynamics (`log_px`),
- and their agreement with the observations (`log_py`).

### Parameters

- `score`  
  Trained score model used inside the diffusion process (score_2, score_4)

- `y`  
  Partial observations used for conditioning.

- `freq` (`'hi'` or `'lo'`)  
  Observation regime:
  - `'hi'`: dense but noisy observations,
  - `'lo'`: sparse but low-noise observations.

- `corrections`  
  Tuple containing the number of Langevin correction steps tested during sampling.

---

In [ ]:
def evaluate(score, y, freq='hi', corrections=(1, 4)):
    chain = make_chain()
    A = lambda x: chain.preprocess(x)[..., :1]

    device = next(score.parameters()).device
    y = y.to(device)

    if freq == 'lo':
        sigma, step = 0.05, 8
    else:
        sigma, step = 0.25, 1

    sde = VPSDE(
        GaussianScore(
            y=y,
            A=lambda x: x[..., ::step, :1],
            std=sigma,
            sde=VPSDE(score, shape=()),
            gamma=3e-2,
        ),
        shape=(65, 3),
    ).to(device)

    results = {}
    for C in corrections:
        x = sde.sample((512,), steps=64, corrections=C, tau=0.25).cpu()
        x = chain.postprocess(x)

        log_px = log_prior(x).mean().item()
        log_py = log_likelihood(y.cpu(), x, A=A, sigma=sigma, step=step).mean().item()
        results[C] = {'log_px': log_px, 'log_py': log_py}

    return results

In [ ]:
freq = 'hi'
y = y_lo if freq == 'lo' else y_hi
score_ = score_2

# accumulation
from collections import defaultdict
acc = defaultdict(lambda: {"log_px": [], "log_py": []})

n_runs = 8  # nombre d'itérations
for _ in range(n_runs):
    idx = torch.randint(0, y.shape[0], (1,)).item()
    results = evaluate(score_, y[idx], freq=freq)
    print(f"Run completed with results: {results}")
    for key, vals in results.items():
        acc[key]["log_px"].append(vals["log_px"])
        acc[key]["log_py"].append(vals["log_py"])

# moyenne
rows = []

for key, vals in acc.items():
    rows.append({
        "correction": key,
        "log_px_mean": sum(vals["log_px"]) / len(vals["log_px"]),
        "log_py_mean": sum(vals["log_py"]) / len(vals["log_py"]),
    })

df = pd.DataFrame(rows).sort_values("correction")

print(df)

Test the evaluation under different configurations: observation type (y_hi or y_lo) and score window size (k = 2 or k = 4). What effects do the number of correction steps (C), the window size (k), and the observation data have on the evaluation scores?